# Comparaison Embeddings × Classifieurs — Sentiment Archelec
Benchmark de plusieurs approches sur les **200 mentions annotées** (2 classes : positif / négatif).

- **Embeddings** : TF-IDF unigrams, TF-IDF bigrams, CamemBERT [CLS] frozen, CamemBERT mean-pooling frozen
- **Classifieurs** : Logistic Regression, SVM, Random Forest
- **Évaluation** : 5-fold cross-validation stratifiée → macro F1 + F1 négatif (classe rare)

## 0 — Imports & données

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score
from sklearn.utils.class_weight import compute_class_weight

ROOT_DIR   = Path(os.getcwd()).parent
INPUT_FILE = ROOT_DIR / "data" / "results" / "output_best_model" / "leaders_mentions.xlsx"
GRAPHS_DIR = ROOT_DIR / "data" / "results" / "output_best_model" / "graphs"
os.makedirs(GRAPHS_DIR, exist_ok=True)

df = pd.read_excel(INPUT_FILE)
df = df[df["sentiment_president"].isin(["positif", "négatif"])].copy()
df["label"] = (df["sentiment_president"] == "positif").astype(int)

X = df["text"].fillna("").tolist()
y = df["label"].tolist()

print(f"Samples : {len(df)}")
print(f"positif : {sum(y)}  |  négatif : {len(y)-sum(y)}")


## 1 — Méthodes TF-IDF + classifieurs classiques

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "f1_macro":   make_scorer(f1_score, average="macro",    zero_division=0),
    "f1_negatif": make_scorer(f1_score, pos_label=0, average="binary", zero_division=0),
    "f1_positif": make_scorer(f1_score, pos_label=1, average="binary", zero_division=0),
}

def run_cv(name, pipeline):
    scores = cross_validate(pipeline, X, y, cv=CV, scoring=scoring, n_jobs=-1)
    return {
        "Méthode":       name,
        "F1 macro":      round(scores["test_f1_macro"].mean(),   3),
        "F1 négatif":    round(scores["test_f1_negatif"].mean(), 3),
        "F1 positif":    round(scores["test_f1_positif"].mean(), 3),
        "F1 macro std":  round(scores["test_f1_macro"].std(),    3),
    }

results = []

# ── TF-IDF unigrams ────────────────────────────────────────────────────────────
tfidf_uni = TfidfVectorizer(max_features=10000, ngram_range=(1,1), sublinear_tf=True)
tfidf_bi  = TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)

classifiers = {
    "LogReg": LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0),
    "SVM":    LinearSVC(class_weight="balanced", max_iter=2000, C=1.0),
    "RF":     RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
}

for clf_name, clf in classifiers.items():
    results.append(run_cv(f"TF-IDF uni + {clf_name}",    Pipeline([("tfidf", tfidf_uni), ("clf", clf)])))
    results.append(run_cv(f"TF-IDF bigram + {clf_name}", Pipeline([("tfidf", tfidf_bi),  ("clf", clf)])))
    print(f"✓ TF-IDF uni & bigram + {clf_name}")

df_tfidf = pd.DataFrame(results)
print()
print(df_tfidf.sort_values("F1 macro", ascending=False).to_string(index=False))


## 2 — CamemBERT frozen embeddings

In [ ]:
import torch
from transformers import CamembertTokenizer, CamembertModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
model_emb = CamembertModel.from_pretrained("camembert-base").to(device)
model_emb.eval()

def get_embeddings(texts, batch_size=32, pooling="cls"):
    """Extrait les embeddings CamemBERT (frozen). pooling : 'cls' ou 'mean'."""
    all_embs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Embed ({pooling})"):
            batch = texts[i:i+batch_size]
            enc = tokenizer(
                batch, truncation=True, padding=True,
                max_length=256, return_tensors="pt"
            ).to(device)
            out = model_emb(**enc)
            if pooling == "cls":
                emb = out.last_hidden_state[:, 0, :]          # [CLS] token
            else:
                mask = enc["attention_mask"].unsqueeze(-1).float()
                emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)  # mean pooling
            all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

X_cls  = get_embeddings(X, pooling="cls")
X_mean = get_embeddings(X, pooling="mean")
print(f"Shape CLS  : {X_cls.shape}")
print(f"Shape Mean : {X_mean.shape}")


In [ ]:
from sklearn.preprocessing import StandardScaler

results_emb = []

for emb_name, X_emb in [("CamemBERT [CLS]", X_cls), ("CamemBERT mean", X_mean)]:
    for clf_name, clf in classifiers.items():
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
        scores = cross_validate(pipe, X_emb, y, cv=CV, scoring=scoring, n_jobs=-1)
        results_emb.append({
            "Méthode":      f"{emb_name} + {clf_name}",
            "F1 macro":     round(scores["test_f1_macro"].mean(),   3),
            "F1 négatif":   round(scores["test_f1_negatif"].mean(), 3),
            "F1 positif":   round(scores["test_f1_positif"].mean(), 3),
            "F1 macro std": round(scores["test_f1_macro"].std(),    3),
        })
        print(f"✓ {emb_name} + {clf_name}")

df_emb = pd.DataFrame(results_emb)
print()
print(df_emb.sort_values("F1 macro", ascending=False).to_string(index=False))


## 3 — Tableau comparatif complet

In [ ]:
# Ajouter CamemBERT fine-tuné (résultat du notebook principal)
df_finetuned = pd.DataFrame([{
    "Méthode":      "CamemBERT fine-tuné",
    "F1 macro":     0.47,
    "F1 négatif":   0.00,
    "F1 positif":   0.95,
    "F1 macro std": float("nan"),
}])

df_all = pd.concat([df_tfidf, df_emb, df_finetuned], ignore_index=True)
df_all = df_all.sort_values("F1 macro", ascending=False).reset_index(drop=True)

print("=" * 70)
print("Comparaison complète — 5-fold CV stratifiée (sauf fine-tuné)")
print("=" * 70)
print(df_all.to_string(index=False))


In [ ]:
# ── Graphique comparatif ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 macro
df_plot = df_all.sort_values("F1 macro")
colors  = ["#e74c3c" if "fine-tuné" in m else "#3498db" for m in df_plot["Méthode"]]
axes[0].barh(df_plot["Méthode"], df_plot["F1 macro"], color=colors, edgecolor="white")
axes[0].axvline(x=0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("F1 macro (↑ meilleur)")
axes[0].set_xlabel("F1 macro")
axes[0].set_xlim(0, 1)

# F1 négatif (classe rare)
df_plot2 = df_all.sort_values("F1 négatif")
colors2  = ["#e74c3c" if "fine-tuné" in m else "#2ecc71" for m in df_plot2["Méthode"]]
axes[1].barh(df_plot2["Méthode"], df_plot2["F1 négatif"], color=colors2, edgecolor="white")
axes[1].axvline(x=0.5, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("F1 négatif — classe rare (↑ meilleur)")
axes[1].set_xlabel("F1 négatif")
axes[1].set_xlim(0, 1)

plt.suptitle("Comparaison Embeddings × Classifieurs — Sentiment Archelec", fontsize=13)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / "comparison_methods.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Graphique sauvegardé : {GRAPHS_DIR / 'comparison_methods.png'}")


In [ ]:
# ── Meilleure méthode ─────────────────────────────────────────────────────────
best = df_all.iloc[0]
print(f"Meilleure méthode (F1 macro) : {best['Méthode']}")
print(f"  F1 macro   : {best['F1 macro']}")
print(f"  F1 négatif : {best['F1 négatif']}")
print(f"  F1 positif : {best['F1 positif']}")

df_all.to_excel(
    ROOT_DIR / "data" / "results" / "output_best_model" / "comparison_methods.xlsx",
    index=False
)
print("\nTableau sauvegardé : comparison_methods.xlsx")
